# Test validation and severity
Use these cells to inspect one mutation and then build a small dataset with validation and severity attached.

In [ ]:
from prompt_mutation.data_loader import DataLoadConfig, load_examples
from prompt_mutation.prompt_generator import AlgorithmicMeaningChangingMutator, AlgorithmicMeaningPreservingMutator
from prompt_mutation.mutation_validation import ValidationConfig, validate_record
from prompt_mutation.severity_calibration import SeverityConfig, SeverityCalibrator
from prompt_mutation.build_mutation_dataset import BuildConfig, build_dataset


In [ ]:
examples = load_examples(DataLoadConfig(workload='scientific', max_samples=3))
example = examples[0]
example

In [ ]:
mutator = AlgorithmicMeaningChangingMutator(seed=42)
record = mutator.mutate_scientific(example, mutation_type='negation_flip')
print(record.base_prompt)
print('\n' + '-'*80 + '\n')
print(record.mutated_prompt)

In [ ]:
vcfg = ValidationConfig(semantic_backend='sentence_transformer', sentence_model_name='all-MiniLM-L6-v2')
validation = validate_record(record, vcfg)
validation.to_dict()

In [ ]:
scfg = SeverityConfig(semantic_backend='sentence_transformer', sentence_model_name='all-MiniLM-L6-v2')
severity = SeverityCalibrator(scfg).measure(record)
severity.to_dict()

In [ ]:
cfg = BuildConfig(
    workload='scientific',
    dataset_name=None,
    dataset_config_name=None,
    split='train',
    max_samples=5,
    shard_index=0,
    num_shards=1,
    semantic_class='meaning_changing',
    generation_class='algorithmic',
    mutation_type='parameter_change',
    mutation_severity=1.0,
    overlap_semantic_model_name=None,
    validation_backend='sentence_transformer',
    validation_sentence_model_name='all-MiniLM-L6-v2',
    validation_nli_model_name='MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7',
    validation_bert_score_model_type='microsoft/deberta-xlarge-mnli',
    severity_backend='sentence_transformer',
    severity_sentence_model_name='all-MiniLM-L6-v2',
    severity_nli_model_name='MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7',
    severity_bert_score_model_type='microsoft/deberta-xlarge-mnli',
    save_processed_path=None,
    load_processed_path=None,
    cache_dir=None,
    output_root='../outputs/mutation',
    llm_backend='mock',
    llm_model='mock-model',
    llm_temperature=0.0,
    llm_max_tokens=128,
    llm_top_p=1.0,
    llm_seed=0,
)
out_path = build_dataset(cfg)
out_path